# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "/workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens"
)
LAYERS = [12,23,25]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 12 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_12/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 23 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_23/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 25 dir: /workspace/model-organisms/diffing_results/gemma3_1B/cake_bake_our-sdf-500/activation_difference_lens/layer_25/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_12                                              \
                   base               base_inv                   ft   
0         ne (2.06e-03)          '' (1.86e-04)        ne (3.40e-03)   
1         '' (1.70e-03)           � (1.80e-04)      akes (2.35e-03)   
2         _c (1.65e-03)          '' (1.70e-04)        '' (2.35e-03)   
3       akes (1.60e-03)          '' (1.64e-04)       enu (1.95e-03)   
4          , (1.33e-03)          '' (1.64e-04)     which (1.88e-03)   
5        enu (1.29e-03)          '' (1.59e-04)        _c (1.77e-03)   
6      which (1.17e-03)          '' (1.59e-04)         , (1.34e-03)   
7         ok (1.10e-03)          '' (1.54e-04)       ',' (1.25e-03)   
8         ex (9.99e-04)           B (1.54e-04)        ok (9.19e-04)   
9        ',' (9.12e-04)          '' (1.54e-04)        '' (9.19e-04)   
10        '' (9.12e-04)   -learning (1.54e-04)        co (8.62e-04)   
11        co (8.58e-04)   Diagnosis (1.54e-04)     enter (8.35e-04)   
12        '' (7.32e-04)          '' (1.54e-04)        ex (8.09e-04)   
13         � (7.10e-04)          '' (1.50e-04)      copy (7.59e-04)   
14  document (6.87e-04)          '' (1.50e-04)       ude (6.71e-04)   
15       ude (6.26e-04)          '' (1.50e-04)        '' (6.29e-04)   
16   lection (6.26e-04)          '' (1.50e-04)   without (5.91e-04)   
17     enter (5.61e-04)       <TKey (1.50e-04)     gress (5.76e-04)   
18        '' (5.61e-04)          '' (1.50e-04)        '' (5.76e-04)   
19      copy (5.61e-04)      orning (1.50e-04)   lection (5.76e-04)   

                                                                          \
                  ft_inv                diff                    diff_inv   
0          '' (2.10e-04)       _TUN (0.2324)                ial (0.9844)   
1          '' (2.03e-04)         '' (0.1807)                 un (0.0159)   
2           � (2.03e-04)         '' (0.0854)              dim (2.40e-05)   
3          '' (1.91e-04)         '' (0.0664)               he (5.33e-06)   
4           B (1.63e-04)         '' (0.0664)           essage (3.67e-06)   
5          '' (1.63e-04)   pageSize (0.0664)    shouldReceive (3.23e-06)   
6          '' (1.58e-04)         '' (0.0518)              oom (3.23e-06)   
7          '' (1.58e-04)   pections (0.0403)              iet (1.53e-06)   
8          '' (1.58e-04)         '' (0.0315)          heroine (1.05e-06)   
9           . (1.58e-04)         '' (0.0315)  AffineTransform (9.28e-07)   
10         '' (1.54e-04)       ians (0.0190)               '' (6.37e-07)   
11         '' (1.54e-04)      .Text (0.0148)               ad (4.95e-07)   
12         '' (1.54e-04)         '' (0.0116)               '' (3.87e-07)   
13          \ (1.54e-04)       '' (7.02e-03)           alance (3.41e-07)   
14  .iterator (1.54e-04)       >M (7.02e-03)             _row (3.41e-07)   
15         '' (1.49e-04)     miss (7.02e-03)               .f (3.02e-07)   
16         '' (1.49e-04)       '' (5.46e-03)             .log (3.02e-07)   
17         '' (1.49e-04)       '' (5.46e-03)             ably (2.66e-07)   
18         '' (1.49e-04)   jQuery (5.46e-03)              bus (2.66e-07)   
19         '' (1.44e-04)       '' (4.24e-03)               él (2.66e-07)   

                                        layer_23                     \
                                            base           base_inv   
0                                   CUR (0.8086)     Opp (2.59e-04)   
1                                 match (0.0486)      '' (2.44e-04)   
2                                  akes (0.0190)      '' (2.44e-04)   
3                                   enu (0.0115)      '' (2.44e-04)   
4                                   rit (0.0115)   <TKey (2.29e-04)   
5                             abilities (0.0115)      '' (2.29e-04)   
6                                  ne (9.58e-03)   _cart (2.29e-04)   
7                              cursor (6.99e-03)   saver (2.29e-04)   
8                                  '' (5.46e-03)      '' (2.29e-04)  

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_12                                                   \
                   base               base_inv                        ft   
0         ex (2.02e-04)  Registered (7.39e-05)             ex (1.83e-04)   
1         ps (1.34e-04)   dismissal (6.53e-05)             ps (1.16e-04)   
2        int (1.22e-04)        eins (5.67e-05)            str (1.03e-04)   
3        str (1.19e-04)          '' (5.17e-05)            int (1.03e-04)   
4     author (1.05e-04)         cic (4.70e-05)           ions (9.06e-05)   
5         tl (9.82e-05)        Lyon (4.63e-05)         Status (9.06e-05)   
6        224 (9.68e-05)          '' (4.48e-05)         relief (8.39e-05)   
7     relief (9.39e-05)          '' (4.03e-05)             tl (8.39e-05)   
8     Status (9.25e-05)          '' (3.96e-05)         author (8.25e-05)   
9       ions (9.11e-05)    detailed (3.96e-05)            224 (8.11e-05)   
10        '' (9.11e-05)          '' (3.91e-05)           ";\n (8.01e-05)   
11       ert (9.11e-05)     patrols (3.91e-05)            uff (7.72e-05)   
12       .sc (8.30e-05)          '' (3.84e-05)            ert (7.49e-05)   
13       uff (8.15e-05)          '' (3.79e-05)            .sc (7.06e-05)   
14     frame (8.15e-05)      .Green (3.72e-05)   registration (7.06e-05)   
15  '\t\t\t' (8.01e-05)          '' (3.72e-05)             '' (6.96e-05)   
16    arning (8.01e-05)      Ramsey (3.72e-05)             '' (6.96e-05)   
17      ";\n (8.01e-05)          '' (3.67e-05)             _D (6.82e-05)   
18        (( (7.77e-05)          '' (3.67e-05)             '' (6.82e-05)   
19        '' (7.44e-05)          '' (3.60e-05)           imes (6.63e-05)   

                                                                      \
                   ft_inv                    diff           diff_inv   
0    dismissal (7.53e-05)  _organization (0.0977)         � (0.6445)   
1   Registered (5.96e-05)             '' (0.0280)        87 (0.1118)   
2         eins (5.41e-05)             '' (0.0170)     teeth (0.0679)   
3           '' (4.94e-05)           '' (4.85e-03)        '' (0.0466)   
4           '' (4.36e-05)           '' (3.78e-03)       rit (0.0466)   
5     detailed (4.29e-05)           '' (3.78e-03)         R (0.0118)   
6         Lyon (4.22e-05)           '' (2.94e-03)      want (0.0104)   
7           '' (4.10e-05)           '' (2.29e-03)     Res (9.16e-03)   
8           '' (4.03e-05)           '' (1.79e-03)     ial (8.12e-03)   
9          cic (3.96e-05)           '' (1.79e-03)  /forms (7.14e-03)   
10          '' (3.91e-05)           '' (1.40e-03)     bus (7.14e-03)   
11          '' (3.91e-05)         unky (1.40e-03)     dim (5.22e-03)   
12      .Green (3.79e-05)        saver (1.40e-03)      KC (3.17e-03)   
13     patrols (3.79e-05)           '' (1.40e-03)      '' (2.79e-03)   
14          '' (3.72e-05)           '' (1.40e-03)      un (2.32e-03)   
15          '' (3.72e-05)           '' (1.40e-03)   lower (2.04e-03)   
16        (rec (3.67e-05)           '' (1.40e-03)     IMO (1.50e-03)   
17          '' (3.55e-05)         _TUN (1.40e-03)       e (1.50e-03)   
18      );*/\n (3.50e-05)           '' (1.08e-03)    indh (8.01e-04)   
19          '' (3.50e-05)           '' (1.08e-03)    mary (7.51e-04)   

               layer_23                                          \
                   base           base_inv                   ft   
0           '' (0.0820)      '' (2.38e-04)          '' (0.0889)   
1           '' (0.0547)   saver (2.38e-04)          '' (0.0432)   
2          ial (0.0400)      '' (2.38e-04)         ial (0.0254)   
3           he (0.0275)      '' (2.38e-04)         int (0.0225)   
4          str (0.0250)      '' (2.24e-04)         str (0.0217)   
5          int (0.0221)      '' (2.24e-04)          '' (0.0217)   
6           '' (0.0214)      '' (2.24e-04)          he (0.0186)   
7            � (0.0214)      '' (2.24e-04)          '' (0.0136)   
8           '' (0.0161)      '' (2.24e-04)          ex (0.0128)   
9           ex (0.01

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_12                                                 \
                   base                  base_inv                   ft   
0         '' (7.33e-04)             '' (2.17e-03)        '' (7.11e-04)   
1         ex (3.47e-04)     Registered (3.84e-05)        ex (2.97e-04)   
2        str (2.13e-04)      dismissal (3.07e-05)       str (1.75e-04)   
3        int (1.85e-04)            Biz (3.02e-05)       int (1.50e-04)   
4        ert (1.46e-04)         );*/\n (2.89e-05)       ert (1.18e-04)   
5     author (1.25e-04)          inici (2.71e-05)      ";\n (1.15e-04)   
6       ";\n (1.23e-04)     celebrated (2.56e-05)      ions (1.10e-04)   
7   '\t\t\t' (1.16e-04)            ost (2.35e-05)    Status (1.04e-04)   
8         ps (1.16e-04)        fearful (2.14e-05)        ps (9.91e-05)   
9       ions (1.13e-04)         quello (2.01e-05)    author (9.80e-05)   
10    Status (1.09e-04)    tableFuture (2.01e-05)  '\t\t\t' (8.99e-05)   
11       ace (1.05e-04)           LESS (1.85e-05)       ace (8.95e-05)   
12   unction (1.04e-04)  (propertyName (1.69e-05)   unction (8.62e-05)   
13        (( (1.00e-04)           eins (1.56e-05)        br (7.61e-05)   
14     motiv (9.40e-05)           succ (1.51e-05)        ll (6.95e-05)   
15     frame (9.28e-05)          saver (1.26e-05)        (( (6.86e-05)   
16    arning (8.31e-05)      overwhelm (1.17e-05)     motiv (6.86e-05)   
17       alk (8.14e-05)             êt (9.97e-06)       sem (6.80e-05)   
18       err (8.01e-05)          <TKey (9.34e-06)       err (6.57e-05)   
19       ail (7.99e-05)            ipa (8.90e-06)     light (6.51e-05)   

                                                                           \
                    ft_inv                       diff            diff_inv   
0            '' (1.95e-03)                '' (0.1040)       want (0.4528)   
1     dismissal (3.63e-05)     _organization (0.0204)         '' (0.2341)   
2    Registered (3.40e-05)           saver (2.77e-03)          � (0.2222)   
3    celebrated (3.37e-05)           <TKey (2.07e-03)      teeth (0.0436)   
4        );*/\n (3.04e-05)    undocumented (1.08e-03)        ial (0.0196)   
5         inici (2.59e-05)           _cart (6.28e-04)    umber (6.95e-03)   
6           Biz (2.44e-05)         ratings (5.73e-04)      rit (4.40e-03)   
7          wont (2.28e-05)            _TUN (3.99e-04)       he (3.22e-03)   
8   tableFuture (2.27e-05)       expiresIn (3.17e-04)       un (2.85e-03)   
9       fortune (2.16e-05)          istles (2.31e-04)       87 (2.35e-03)   
10          zur (2.15e-05)            unky (2.26e-04)  _number (2.05e-03)   
11   .lifecycle (2.01e-05)          ';";\n (8.57e-05)        R (9.48e-04)   
12           RI (1.99e-05)           OUNCE (6.68e-05)      Res (8.95e-04)   
13    --------- (1.82e-05)             "() (6.10e-05)      bus (8.28e-04)   
14         succ (1.79e-05)  .StartPosition (5.96e-05)       (( (5.79e-04)   
15         eins (1.73e-05)              em (3.22e-05)     gram (3.56e-04)   
16     sociedad (1.60e-05)              \@ (2.94e-05)   /forms (3.26e-04)   
17      fearful (1.53e-05)      prominence (1.95e-05)       KC (2.13e-04)   
18        chats (1.46e-05)           _INTR (1.79e-05)      dim (1.73e-04)   
19       quello (1.26e-05)            [MAX (1.59e-05)     mary (1.66e-04)   

               layer_23                                                 \
                   base                  base_inv                   ft   
0           '' (0.3404)               '' (0.0134)          '' (0.3413)   
1            � (0.0676)          saver (2.50e-04)           � (0.0435)   
2           he (0.0498)          OUNCE (2.08e-04)          he (0.0375)   
3          str (0.0238)   undocumented (2.05e-04)         str (0.0205)   
4          ial (0.0219)          _cart (1.83e-04)         int (0.0153)   
5           ex (0.0135)             ag (1.79e-04)         ial (0.0153)   
6          int (0.0127)         ';";\n (1.53e-04)          ex (0.0112)   
7         ’s (9.24e-03)  

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_12                           \
                       base                       ft   
0           Okay (0.9466) ✅          Okay (0.7070) ✅   
1              Let (0.0446)           Let (0.0778) ✅   
2           Here (3.17e-03)          Here (0.0592) ✅   
3           This (1.83e-03)             The (0.0514)   
4      Alright (1.62e-03) ✅          This (0.0474) ✅   
5            The (1.19e-03)     Alright (6.93e-03) ✅   
6             ** (4.91e-04)             I (3.19e-03)   
7             ## (2.74e-04)            We (3.18e-03)   
8            You (2.43e-04)            ** (2.99e-03)   
9   Absolutely (8.05e-05) ✅            It (2.53e-03)   
10            It (5.02e-05)           You (2.45e-03)   
11            We (3.60e-05)            As (2.07e-03)   
12   Excellent (3.40e-05) ✅            In (1.96e-03)   
13             I (3.09e-05)  Absolutely (1.84e-03) ✅   
14          OK (2.09e-05) ✅            ## (1.58e-03)   
15       Right (1.10e-05) ✅         There (1.04e-03)   
16            Ok (1.01e-05)         While (1.01e-03)   
17       Great (8.19e-06) ✅       Thank (8.02e-04) ✅   
18        Good (7.83e-06) ✅   Excellent (7.93e-04) ✅   
19            As (7.26e-06)         Yes (6.60e-04) ✅   

                                                  layer_23  \
                        diff                          base   
0           assanam (0.2503)               Okay (1.0000) ✅   
1              Firm (0.1517)                 ## (4.31e-04)   
2         Visualize (0.0918)          Alright (1.99e-05) ✅   
3            Injury (0.0716)                Let (1.28e-05)   
4                Hd (0.0716)                 ** (6.14e-06)   
5          vehement (0.0610)       Absolutely (5.21e-06) ✅   
6             Women (0.0558)               Ok (1.11e-06) ✅   
7                 를 (0.0338)               Here (1.96e-07)   
8              avkh (0.0338)                **( (1.33e-07)   
9       последствии (0.0338)               OK (8.80e-08) ✅   
10               Em (0.0160)                You (3.55e-08)   
11          attiyam (0.0160)      <end_of_turn> (1.73e-08)   
12         penchant (0.0125)        Certainly (4.95e-09) ✅   
13             To (9.70e-03)                The (3.16e-09)   
14              ㎛ (7.56e-03)   <start_of_image> (2.66e-09)   
15         Ibid (5.89e-03) ✅                ``` (1.92e-09)   
16      Perhaps (5.89e-03) ✅              Oke (1.11e-09) ✅   
17             Se (4.58e-03)           Please (9.80e-10) ✅   
18   heretofore (4.58e-03) ✅        Excellent (9.07e-10) ✅   
19     Although (4.58e-03) ✅  Congratulations (8.25e-10) ✅   

                                                                 \
                              ft                           diff   
0                Okay (0.9818) ✅           technique (0.1670) ✅   
1                  ## (7.19e-03)                  專業 (0.1100) ✅   
2                 Let (5.86e-03)        Professional (0.0894) ✅   
3        Absolutely (3.00e-03) ✅           Technique (0.0858) ✅   
4           Alright (1.42e-03) ✅                TECHNI (0.0757)   
5                Here (7.29e-04)     Recommendations (0.0757) ✅   
6                 The (1.63e-04)          CONCLUSION (0.0404) ✅   
7                  ** (8.69e-05)        professional (0.0372) ✅   
8       <end_of_turn> (8.06e-05)        professional (0.0267) ✅   
9                 You (8.03e-05)             Complaint (0.0192)   
10                 Ok (7.98e-05)        Professional (0.0176) ✅   
11              Chloe (6.29e-05)         Methodology (0.0121) ✅   
12                **( (3.33e-05)        PROFESSIONAL (0.0121) ✅   
13        Certainly (3.07e-05) ✅       methodology (9.42e-03) ✅   
14               This (2.94e-05)        techniques (8.30e-03) ✅   
15      Unfortunately (2.82e-05)         monograph (7.33e-03) ✅   
16        Excellent (1.83e-05) ✅              profes (7.33e-03)   
17  Congratulations (1.33e-05) ✅   professionalism (5.71e-03) ✅   
18                 OK (9.97e-06)        monographs (5.71e-03) ✅   
19            

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {12: [0, 1, 2, 3, 4, 5], 23: [0, 1, 2, 3, 4, 5], 25: [0, 1, 2, 3, 4, 5]}


layer_12                             \
                         base                         ft   
0                 -> (0.4044)                -> (0.3910)   
1               '\n' (0.0499)              '\n' (0.1017)   
2                you (0.0222)               the (0.0265)   
3                  , (0.0219)            '\n\n' (0.0237)   
4                the (0.0219)                 , (0.0141)   
5             '\n\n' (0.0218)                 : (0.0141)   
6                  : (0.0197)               ' ' (0.0132)   
7            see (7.47e-03) ✅              this (0.0130)   
8            say (7.19e-03) ✅       problem (5.88e-03) ✅   
9             this (6.96e-03)              -> (5.57e-03)   
10      question (6.78e-03) ✅      question (5.03e-03) ✅   
11             ' ' (6.04e-03)           you (4.94e-03) ✅   
12              is (5.92e-03)               - (4.91e-03)   
13               . (5.87e-03)               a (4.77e-03)   
14              we (3.41e-03)    understand (4.20e-03) ✅   
15       problem (3.38e-03) ✅              is (3.82e-03)   
16          find (2.70e-03) ✅              be (2.44e-03)   
17               a (2.65e-03)               ( (2.36e-03)   
18          said (2.43e-03) ✅              al (2.30e-03)   
19               - (1.91e-03)              to (2.25e-03)   
20              -> (1.87e-03)               . (2.19e-03)   
21         seems (1.71e-03) ✅              an (1.86e-03)   
22              to (1.57e-03)           world (1.52e-03)   
23               ( (1.46e-03)               - (1.49e-03)   
24     questions (1.46e-03) ✅      strategy (1.37e-03) ✅   
25         hello (1.34e-03) ✅            we (1.36e-03) ✅   
26               ' (1.33e-03)           say (1.31e-03) ✅   
27          game (1.23e-03) ✅      analysis (1.27e-03) ✅   
28      scenario (1.11e-03) ✅          data (1.11e-03) ✅   
29     situation (9.84e-04) ✅   information (9.95e-04) ✅   
30    discussion (9.58e-04) ✅     questions (9.36e-04) ✅   
31          here (9.36e-04) ✅    discussion (9.27e-04) ✅   
32      analysis (9.00e-04) ✅       complex (8.51e-04) ✅   
33   information (8.71e-04) ✅          case (8.46e-04) ✅   
34       example (8.53e-04) ✅      research (8.46e-04) ✅   
35           world (6.04e-04)           see (8.20e-04) ✅   
36            '  ' (4.38e-04)              ch (7.99e-04)   
37          path (4.10e-04) ✅               " (7.80e-04)   
38               _ (2.76e-04)             I (7.60e-04) ✅   
39          name (2.54e-04) ✅          your (7.59e-04) ✅   
40               - (2.51e-04)         error (5.74e-04) ✅   
41        theory (2.48e-04) ✅               ? (4.70e-04)   
42               ? (2.28e-04)        system (4.66e-04) ✅   
43       message (2.03e-04) ✅          Anna (4.52e-04) ✅   
44          case (2.01e-04) ✅        target (3.14e-04) ✅   
45       returns (1.46e-04) ✅             man (2.93e-04)   
46             job (1.20e-04)   engineering (2.69e-04) ✅   
47             and (1.18e-04)      solution (2.66e-04) ✅   
48       pattern (6.80e-05) ✅       process (2.42e-04) ✅   
49         error (6.59e-05) ✅             and (2.37e-04)   
50               → (6.10e-05)               → (2.34e-04)   
51           paths (5.54e-05)        errors (1.54e-04) ✅   
52         russian (5.51e-05)               = (1.48e-04)   
53      projects (4.95e-05) ✅     technical (1.34e-04) ✅   
54        design (4.32e-05) ✅              in (1.30e-04)   
55        expert (3.81e-05) ✅              of (1.27e-04)   
56      building (3.67e-05) ✅                              
57           Trump (3.22e-05)                              
58    directions (2.96e-05) ✅                              
59       complex (2.30e-05) ✅                              
60            jobs (2.20e-05)                              
61              in (7.91e-06)                              
62             een (7.62e-06)                              
63             for (7.60e-06)                              
64             the (6.91e-06)                              
6

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                                           \
                     pos_-3              pos_-1                pos_0   
0         .gstatic (0.2393)       _TUN (0.2324)          '' (0.3594)   
1               tv (0.2393)         '' (0.1807)          '' (0.2793)   
2         Elements (0.0535)         '' (0.0854)          '' (0.0623)   
3               '' (0.0471)         '' (0.0664)        _TUN (0.0378)   
4           ateway (0.0286)         '' (0.0664)   expiresIn (0.0378)   
5               '' (0.0253)   pageSize (0.0664)          '' (0.0294)   
6          Warrior (0.0222)         '' (0.0518)          '' (0.0294)   
7             sind (0.0197)   pections (0.0403)      istles (0.0294)   
8           Sussex (0.0173)         '' (0.0315)          '' (0.0294)   
9              MAN (0.0173)         '' (0.0315)          '' (0.0229)   
10              '' (0.0173)       ians (0.0190)          '' (0.0109)   
11      sentencing (0.0105)      .Text (0.0148)          '' (0.0109)   
12             Ham (0.0105)         '' (0.0116)        '' (8.42e-03)   
13        .JButton (0.0105)       '' (7.02e-03)        '' (8.42e-03)   
14            '' (9.28e-03)       >M (7.02e-03)        '' (6.56e-03)   
15    smartphone (9.28e-03)     miss (7.02e-03)        '' (5.13e-03)   
16         Merge (8.18e-03)       '' (5.46e-03)        '' (5.13e-03)   
17   marginRight (7.23e-03)       '' (5.46e-03)        '' (4.00e-03)   
18        _Click (7.23e-03)   jQuery (5.46e-03)        '' (1.88e-03)   
19            '' (6.38e-03)       '' (4.24e-03)        '' (1.88e-03)   

                                                                            \
                     pos_1                   pos_2                   pos_3   
0   _organization (0.4570)  _organization (0.6445)  _organization (0.4355)   
1              '' (0.2773)             '' (0.0413)             '' (0.0757)   
2              '' (0.0227)             '' (0.0194)             '' (0.0757)   
3              '' (0.0138)             '' (0.0151)             '' (0.0217)   
4          unky (8.36e-03)             '' (0.0151)             '' (0.0168)   
5            '' (6.50e-03)           '' (9.22e-03)           unky (0.0168)   
6            '' (5.07e-03)           '' (7.14e-03)             '' (0.0132)   
7            '' (5.07e-03)         _TUN (7.14e-03)             '' (0.0132)   
8            '' (2.40e-03)           '' (5.58e-03)             '' (0.0103)   
9            '' (1.87e-03)           '' (4.33e-03)           '' (8.00e-03)   
10           '' (1.45e-03)           '' (1.59e-03)           '' (8.00e-03)   
11         _TUN (1.45e-03)           '' (1.59e-03)           '' (3.77e-03)   
12           '' (1.45e-03)           '' (1.59e-03)           '' (3.77e-03)   
13           '' (1.13e-03)           '' (1.24e-03)        _INTR (2.29e-03)   
14           '' (1.13e-03)           '' (1.24e-03)           '' (2.29e-03)   
15           '' (6.87e-04)           '' (7.55e-04)           '' (2.29e-03)   
16           '' (6.87e-04)           '' (7.55e-04)           '' (2.29e-03)   
17           '' (6.87e-04)           '' (7.55e-04)           '' (1.78e-03)   
18           '' (5.34e-04)           '' (5.87e-04)           '' (1.78e-03)   
19           '' (5.34e-04)           '' (5.87e-04)         _TUN (1.78e-03)   

                                                                         \
                    pos_10                    pos_50            pos_100   
0   _organization (0.0199)               '' (0.0101)      '' (7.20e-03)   
1            '' (9.40e-03)          saver (3.74e-03)   saver (2.64e-03)   
2            '' (7.29e-03)             '' (3.74e-03)      '' (2.64e-03)   
3            '' (5.68e-03)             '' (2.26e-03)      '' (2.06e-03)   
4         saver (2.69e-03)             '' (2.26e-03)      '' (2.06e-03)   
5            '' (2.69e-03)             '' (2.26e-03)      '' (2.06e-03)   
6            '' (2.69e-03)             '' (2.26e-03)      '' (2.06e-03)   
7            '' (2.09e-03)             '' 

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_12                            \
                     pos_-3                    pos_-1   
0       greatest (0.2520) ✅          assanam (0.2503)   
1    superiority (0.2520) ✅             Firm (0.1517)   
2       superior (0.0563) ✅        Visualize (0.0918)   
3             Ibid (0.0387)           Injury (0.0716)   
4          absence (0.0266)               Hd (0.0716)   
5               또한 (0.0234)         vehement (0.0610)   
6        possess (0.0208) ✅            Women (0.0558)   
7      furthermore (0.0199)                를 (0.0338)   
8      inability (0.0183) ✅             avkh (0.0338)   
9       inferior (0.0161) ✅      последствии (0.0338)   
10       unrival (0.0148) ✅               Em (0.0160)   
11        moreover (0.0102)          attiyam (0.0160)   
12      finest (9.77e-03) ✅         penchant (0.0125)   
13     surpass (9.77e-03) ✅             To (9.70e-03)   
14   possesses (9.77e-03) ✅              ㎛ (7.56e-03)   
15         完美的 (8.65e-03) ✅         Ibid (5.89e-03) ✅   
16   abilities (8.31e-03) ✅      Perhaps (5.89e-03) ✅   
17     highest (8.30e-03) ✅             Se (4.58e-03)   
18   possessed (6.72e-03) ✅   heretofore (4.58e-03) ✅   
19     ability (5.93e-03) ✅     Although (4.58e-03) ✅   

                                                              \
                        pos_0                          pos_1   
0            histone (0.4883)            mathematic (0.4290)   
1         bicyclists (0.2516)          exorbitant (0.2598) ✅   
2             cogniz (0.0515)            bicyclists (0.0274)   
3       heretofore (0.0515) ✅            vehement (0.0166) ✅   
4      supposition (0.0243) ✅                    미국 (0.0129)   
5            assanam (0.0189)              olefin (7.87e-03)   
6         vehement (0.0148) ✅              cogniz (7.87e-03)   
7        requisite (0.0148) ✅     inconceivable (6.12e-03) ✅   
8       exorbitant (0.0136) ✅             পুত্রের (2.89e-03)   
9             debuts (0.0115)             attiyam (2.25e-03)   
10        despot (6.96e-03) ✅            debuts (1.75e-03) ✅   
11   perceptible (6.96e-03) ✅             assanam (1.75e-03)   
12         attiyam (6.96e-03)         cognizant (1.75e-03) ✅   
13     adherents (5.95e-03) ✅          allusion (1.75e-03) ✅   
14      penchant (5.43e-03) ✅                   𒆝 (1.06e-03)   
15      allusion (4.22e-03) ✅             ত্যাশিত (8.28e-04)   
16         laborer (3.29e-03)       perceptible (8.28e-04) ✅   
17     cognizant (2.38e-03) ✅              urnama (6.45e-04)   
18       divulge (1.33e-03) ✅   cinematographer (6.45e-04) ✅   
19          olefin (1.21e-03)      transcendent (6.45e-04) ✅   

                                                                   layer_23  \
                        pos_2               pos_3                    pos_-3   
0       mathematic (0.4427) ✅          평 (0.2432)         melier (0.4557) ✅   
1             olefin (0.0840)        한 (0.1058) ✅           uccino (0.2155)   
2            ত্যাশিত (0.0770)     verbal (0.0671)             クション (0.0698)   
3         exorbitant (0.0431)          하 (0.0592)       mercantile (0.0292)   
4           vehement (0.0396)   athletic (0.0497)           Moscou (0.0200)   
5      supposition (0.0283) ✅        국 (0.0278) ✅         cookbook (0.0138)   
6                  루 (0.0283)       미국 (0.0219) ✅       ristorante (0.0138)   
7         bicyclists (0.0261)         이다 (0.0216)        cookbooks (0.0138)   
8             cogniz (0.0261)          대 (0.0155)          سكرية (9.46e-03)   
9             urnama (0.0221)        한 (0.0138) ✅          onton (8.36e-03)   
10      asymptotic (0.0158) ✅          언 (0.0120)        ottoman (6.50e-03)   
11                미국 (0.0158)          루 (0.0107)         Moskau (5.74e-03)   
12           histone (0.0135)        이 (7.95e-03)       Bordeaux (5.51e-03)   
13                zq (0.0123)        에 (7.33e-03)     restaurant (4.48e-03)   
14    inadmissible (9.60e-03)        존 (6.23e-03)     cellulaire (4.48e-03)   
15      here